# Research Questions: Prediksi Kandungan Organik Tanah

Notebook ini menjawab sepuluh pertanyaan riset tentang prediksi kandungan organik tanah. Fokus analisisnya adalah urgensi masalah, pola geografis dan ekologis, evaluasi model, feature engineering, pemilihan metrik, serta peluang penggunaan data eksternal.

Hasil yang ditampilkan menggunakan data training yang tersedia. Interpretasi statistik diperlakukan sebagai asosiasi, bukan bukti kausal, sehingga temuan model tetap perlu dibaca bersama konteks agronomi dan proses sampling.


## Setup, data, dan fungsi bantuan

In [ ]:
import os
import time
import warnings
from itertools import combinations
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str((Path.cwd() / ".mplconfig").resolve())
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display
from scipy import stats
from sklearn.cross_decomposition import PLSRegression
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_log_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor

RANDOM_STATE = 42
TARGET = "property_organic_content"
N_FOLDS = 5
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")
GREEN = "#2D6A4F"
LIGHT_GREEN = "#74C69D"
DARK_GREEN = "#1B4332"
ORANGE = "#F4A261"
RED = "#C1121F"
BLUE = "#457B9D"
plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

train = pd.read_csv(DATA_DIR / "train.csv")
y = train[TARGET].copy()

assert TARGET in train.columns
assert train["sample_id"].is_unique
print(f"Train shape: {train.shape}")
print(f"Target mean: {y.mean():.4f} | median: {y.median():.4f} | skewness: {y.skew():.4f}")


In [ ]:
def regression_metrics(y_true, y_pred):
    y_pred_nonnegative = np.clip(np.asarray(y_pred), 0, None)
    return {
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MedianAE": median_absolute_error(y_true, y_pred),
        "RMSLE": np.sqrt(mean_squared_log_error(y_true, y_pred_nonnegative)),
        "R2": r2_score(y_true, y_pred),
    }


def bootstrap_mean_ci(values, n_boot=500, confidence=0.95, seed=RANDOM_STATE):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return values[0], values[0]
    rng = np.random.default_rng(seed)
    boot_means = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_boot)
    ])
    alpha = 1 - confidence
    return (
        np.quantile(boot_means, alpha / 2),
        np.quantile(boot_means, 1 - alpha / 2),
    )


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def eta_squared(data, group_col, target_col=TARGET):
    grand_mean = data[target_col].mean()
    ss_between = sum(
        len(group) * (group[target_col].mean() - grand_mean) ** 2
        for _, group in data.groupby(group_col)
    )
    ss_total = ((data[target_col] - grand_mean) ** 2).sum()
    return ss_between / ss_total


def make_features(df, stage="full", drop_source=False):
    stage_order = {
        "raw": 0,
        "missingness": 1,
        "spectral": 2,
        "soil": 3,
        "geographic": 4,
        "full": 5,
    }
    if stage not in stage_order:
        raise ValueError(f"Stage tidak dikenal: {stage}")

    level = stage_order[stage]
    X = df.drop(columns=[TARGET, "sample_id"], errors="ignore").copy()
    if drop_source:
        X = X.drop(columns=["source_id"], errors="ignore")

    band_a = [c for c in X.columns if c.startswith("spectral_band_A_PC_")]
    band_b = [c for c in X.columns if c.startswith("spectral_band_B_PC_")]
    chemistry = [
        "property_acidity_index", "cation_Ca", "cation_Mg",
        "cation_Na", "cation_exchange_capacity",
    ]
    eps = 1e-6

    if level >= 1:
        X["missing_total"] = X.isna().sum(axis=1)
        X["chem_missing_count"] = X[chemistry].isna().sum(axis=1)
        X["band_B_available_actual_num"] = X[band_b].notna().any(axis=1).astype(int)
        X["band_B_available_actual"] = np.where(
            X[band_b].notna().any(axis=1), "YES", "NO"
        )
        X["band_B_missing_count"] = X[band_b].isna().sum(axis=1)
        X["coord_available_num"] = X[["latitude", "longitude"]].notna().all(axis=1).astype(int)
        X["coordinates_available"] = np.where(
            X[["latitude", "longitude"]].notna().all(axis=1), "YES", "NO"
        )

    if level >= 2:
        for prefix, cols in [("A", band_a), ("B", band_b)]:
            values = X[cols]
            X[f"{prefix}_mean"] = values.mean(axis=1)
            X[f"{prefix}_std"] = values.std(axis=1)
            X[f"{prefix}_min"] = values.min(axis=1)
            X[f"{prefix}_max"] = values.max(axis=1)
            X[f"{prefix}_l2"] = np.sqrt((values ** 2).sum(axis=1))
            X[f"{prefix}_abs_sum"] = values.abs().sum(axis=1)
            X[f"{prefix}_max_abs"] = values.abs().max(axis=1)
            X[f"{prefix}_pos_count"] = (values > 0).sum(axis=1)
            for col in cols[:3]:
                X[f"{col}_abs"] = X[col].abs()
                X[f"{col}_sq"] = X[col] ** 2

    if level >= 3:
        X["particle_total"] = X["property_particle_coarse"] + X["property_particle_fine"]
        X["fine_fraction"] = X["property_particle_fine"] / (X["particle_total"] + eps)
        X["fine_to_coarse"] = X["property_particle_fine"] / (X["property_particle_coarse"] + eps)
        X["coarse_fine_ratio"] = X["property_particle_coarse"] / (X["property_particle_fine"].abs() + eps)
        X["fine_minus_coarse"] = X["property_particle_fine"] - X["property_particle_coarse"]
        X["base_cation_sum"] = X[["cation_Ca", "cation_Mg", "cation_Na"]].sum(axis=1, min_count=1)
        X["ca_mg_ratio"] = X["cation_Ca"] / (X["cation_Mg"] + eps)
        X["mg_ca_ratio"] = X["cation_Mg"] / (X["cation_Ca"] + eps)
        X["base_saturation_proxy"] = X["base_cation_sum"] / (X["cation_exchange_capacity"] + eps)
        X["base_to_cec_ratio"] = X["base_saturation_proxy"]
        X["ca_cec_ratio"] = X["cation_Ca"] / (X["cation_exchange_capacity"] + eps)
        X["mg_cec_ratio"] = X["cation_Mg"] / (X["cation_exchange_capacity"] + eps)
        X["cec_per_fine"] = X["cation_exchange_capacity"] / (X["property_particle_fine"] + eps)
        X["acidity_x_cec"] = X["property_acidity_index"] * X["cation_exchange_capacity"]
        X["acidity_per_cec"] = X["property_acidity_index"] / (X["cation_exchange_capacity"] + eps)
        for col in ["cation_exchange_capacity", "cation_Ca", "cation_Mg", "property_acidity_index"]:
            X[f"log1p_{col}"] = np.log1p(X[col])

    if level >= 4:
        X["abs_latitude"] = X["latitude"].abs()
        X["abs_longitude"] = X["longitude"].abs()
        X["lat_lon_sum"] = X["latitude"] + X["longitude"]
        X["lat_lon_diff"] = X["latitude"] - X["longitude"]
        X["lat_lon_prod"] = X["latitude"] * X["longitude"]
        X["lat_round_1"] = X["latitude"].round(1)
        X["lon_round_1"] = X["longitude"].round(1)
        X["latlon_grid1"] = X["lat_round_1"].astype("string").fillna("NA") + "|" + X["lon_round_1"].astype("string").fillna("NA")

        def add_interaction(cols, name):
            if all(c in X.columns for c in cols):
                X[name] = X[cols].astype("string").fillna("NA").agg("|".join, axis=1)

        for cols, name in [
            (["geo_zone_macro", "geo_zone_meso", "geo_zone_micro"], "geo_hierarchy"),
            (["biome", "land_cover_type"], "biome_landcover"),
            (["source_id", "geo_zone_micro"], "source_micro"),
            (["source_id", "land_cover_type"], "source_landcover"),
            (["geo_zone_meso", "land_cover_type"], "meso_landcover"),
            (["land_cover_type", "parent_rock_type"], "landcover_rock"),
            (["source_id", "parent_rock_type"], "source_rock"),
            (["source_id", "band_B_available_actual"], "source_bandB"),
            (["geo_zone_macro", "biome"], "macro_biome"),
            (["geo_zone_macro", "land_cover_type"], "macro_landcover"),
        ]:
            add_interaction(cols, name)

    categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

    if level >= 5:
        for col in categorical_cols:
            freq = X[col].astype("string").fillna("NA").value_counts(dropna=False)
            X[f"{col}_freq"] = X[col].astype("string").fillna("NA").map(freq).astype(float)

    X = X.replace([np.inf, -np.inf], np.nan)
    categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    return X, categorical_cols


X_full, cat_cols_full = make_features(train, stage="full")
y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
stratified_cv = StratifiedKFold(
    n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE
)
cv_splits = list(stratified_cv.split(X_full, y_bins))
print(f"Engineered feature shape: {X_full.shape}")
print(f"Categorical features: {len(cat_cols_full)}")


## Ringkasan data sebelum menjawab pertanyaan

Ringkasan ini digunakan untuk memahami representasi data dan potensi bias sampel.

In [ ]:
band_b_cols = [c for c in train.columns if c.startswith("spectral_band_B_PC_")]
overview = pd.Series({
    "jumlah_sampel": len(train),
    "jumlah_fitur": train.shape[1] - 1,
    "jumlah_source": train["source_id"].nunique(),
    "jumlah_zona_macro": train["geo_zone_macro"].nunique(),
    "jumlah_zona_meso": train["geo_zone_meso"].nunique(),
    "jumlah_zona_micro": train["geo_zone_micro"].nunique(),
    "band_B_missing_pct": train[band_b_cols].isna().all(axis=1).mean() * 100,
    "target_mean": y.mean(),
    "target_median": y.median(),
    "target_skewness": y.skew(),
})
display(overview.to_frame("nilai"))

_, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(y, bins=45, kde=True, color=GREEN, ax=axes[0])
axes[0].axvline(y.mean(), color=RED, ls="--", label="Mean")
axes[0].axvline(y.median(), color=ORANGE, ls=":", label="Median")
axes[0].set_title("Distribusi target")
axes[0].legend()

train["biome"].value_counts().sort_values().plot(
    kind="barh", color=LIGHT_GREEN, ax=axes[1]
)
axes[1].set_title("Representasi sampel per bioma")
axes[1].set_xlabel("Jumlah sampel")
plt.tight_layout()
plt.show()

# Soal 1 — Urgensi bisnis dan kebijakan pangan

> **Dalam konteks bisnis pertanian dan kebijakan pangan, apakah memprediksi kandungan organik tanah
> merupakan hal yang mendesak? Jelaskan urgensinya dari perspektif efisiensi biaya operasional dan
> manajemen lahan berkelanjutan!**

## Pendekatan dan asumsi

Jawaban menggabungkan konteks dataset dengan literatur FAO dan IPCC. Karena dataset tidak menyediakan
data biaya, notebook tidak mengarang nominal penghematan. Efisiensi dijelaskan melalui perubahan alur
kerja dan alokasi sumber daya.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis("off")
boxes = [
    (0.03, "Spektrum +\ngeografi", LIGHT_GREEN),
    (0.27, "Prediksi +\nketidakpastian", "#95D5B2"),
    (0.51, "Prioritas sampel\nlaboratorium", ORANGE),
    (0.75, "Tindakan spesifik\nlokasi", "#E9C46A"),
]
for x, label, color in boxes:
    ax.text(
        x, 0.55, label, ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.7", fc=color, ec=DARK_GREEN)
    )
for x1, x2 in [(0.18, 0.26), (0.43, 0.50), (0.67, 0.74)]:
    ax.annotate("", xy=(x2, 0.55), xytext=(x1, 0.55),
                arrowprops=dict(arrowstyle="->", lw=2, color=DARK_GREEN))
ax.text(
    0.5, 0.08,
    "Prinsip: model menyaring dan memprioritaskan; uji laboratorium tetap menjadi konfirmasi.",
    ha="center", fontsize=11, color=DARK_GREEN
)
ax.set_title("Alur keputusan yang menurunkan pengujian merata", pad=15)
plt.show()

## Jawaban Soal 1

**Ya, prediksi kandungan organik tanah mendesak sebagai alat screening dan prioritisasi.**

1. **Efisiensi biaya operasional.** Pengujian laboratorium merata membutuhkan pengambilan sampel,
   transportasi, preparasi, reagen, instrumen, dan waktu analis. Model dapat membuat peta risiko awal,
   sehingga laboratorium difokuskan pada lokasi bernilai ekstrem, berketidakpastian tinggi, atau penting
   secara kebijakan. Penghematan berasal dari *targeted sampling*, bukan dari menghapus laboratorium.
2. **Efisiensi input pertanian.** Kandungan organik berkaitan dengan kesuburan, retensi air, struktur
   tanah, dan jasa ekosistem. Informasi spasial yang lebih cepat mendukung pemberian bahan organik,
   pemupukan, dan konservasi yang spesifik lokasi.
3. **Manajemen berkelanjutan.** Prediksi berulang membantu mendeteksi degradasi dan memprioritaskan
   rehabilitasi. FAO menyatakan bahwa soil organic carbon penting bagi kesehatan, kesuburan, produksi
   pangan, dan jasa ekosistem; IPCC juga menempatkan pengelolaan lahan berkelanjutan sebagai bagian dari
   ketahanan pangan dan mitigasi/adaptasi iklim.
4. **Batas keputusan.** Model tidak mengukur stok karbon secara langsung dan tidak membuktikan sebab
   perubahan. Keputusan berisiko tinggi tetap memerlukan konfirmasi laboratorium, audit error per
   wilayah, dan pembaruan model.

**Kesimpulan:** urgensinya tinggi ketika model dipakai untuk mempercepat dan memprioritaskan tindakan,
bukan ketika diposisikan sebagai pengganti absolut metode referensi.

Referensi: [FAO — Soil Organic Carbon](https://www.fao.org/global-soil-partnership/areas-of-work/soil-organic-carbon/en/);
[IPCC — Climate Change and Land](https://www.ipcc.ch/srccl/).

**Keterbatasan:** dataset tidak memiliki komponen biaya dan tanggal pemantauan, sehingga besaran ROI
serta kemampuan mendeteksi tren waktu belum dapat dihitung.

# Soal 2 - Overfit atau underfit

> **Apakah model prediksi kandungan organik tanah Anda mengalami overfit atau underfit? Tunjukkan
> buktinya secara empiris melalui metrik evaluasi atau melalui visualisasi yang relevan, serta jelaskan
> langkah mitigasi yang Anda lakukan! Jika tidak mengalami keduanya, jelaskan alasannya!**

## Pendekatan dan asumsi

Diagnosis dilakukan dengan membandingkan performa training dan validasi, lalu menguji apakah error tetap stabil ketika validasi dipisahkan berdasarkan `source_id`. Validasi berbasis sumber penting karena data dari sumber/laboratorium yang sama dapat memiliki pola yang mirip, sehingga random split saja bisa terlalu optimis.

Bukti yang digunakan adalah RMSE per fold, residual terhadap prediksi, perbandingan sampel dengan dan tanpa Band B, serta performa beberapa keluarga model sebelum digabungkan.


In [ ]:
V6_CHECKPOINT_PATH = DATA_DIR / "checkpoint_v6.npz"
V6_SUMMARY_PATH = DATA_DIR / "model_summary_v6.csv"
PUBLIC_BEST_SCORE = 11.95464
TAIL_THRESHOLD = 65.0
TAIL_ALPHA = 0.05
FINAL_MODEL_NAME = "Final stacked model + high-tail correction"
RAW_ENSEMBLE_NAME = "Final stacked model before high-tail correction"

assert V6_CHECKPOINT_PATH.exists(), f"Missing {V6_CHECKPOINT_PATH}. Run model training first."
assert V6_SUMMARY_PATH.exists(), f"Missing {V6_SUMMARY_PATH}. Run model training first."

v6_checkpoint = np.load(V6_CHECKPOINT_PATH)
v6_summary = pd.read_csv(V6_SUMMARY_PATH)

base_model_names = [
    name.replace("oof_", "").replace(".npy", "")
    for name in v6_checkpoint.files
    if name.startswith("oof_")
]
base_model_names = [
    name for name in base_model_names
    if f"pred_{name}" in v6_checkpoint.files
]

PRETTY_NAME = {
    "cat_num_d6": "CatBoost depth 6",
    "cat_num_d7_deep": "CatBoost depth 7",
    "ridge_te40": "Ridge with encoded categorical features",
    "lgb_base": "LightGBM",
    "lgb_base_log": "LightGBM log target",
    "lgb_extra_wide": "LightGBM wide",
    "xgb_d4": "XGBoost depth 4",
    "hgb": "HistGradientBoosting",
}

base_oof = {name: v6_checkpoint[f"oof_{name}"] for name in base_model_names}
O_base = np.column_stack([base_oof[name] for name in base_model_names])

y_np = y.to_numpy()
source_counts = train["source_id"].map(train["source_id"].value_counts()).to_numpy()
meta_extra = np.column_stack([
    np.log1p(source_counts),
    X_full["missing_total"].to_numpy(),
])


def apply_tail_shrink(pred, threshold=TAIL_THRESHOLD, alpha=TAIL_ALPHA):
    pred = np.asarray(pred, dtype=float)
    return np.clip(pred - alpha * np.maximum(pred - threshold, 0), 0, None)


def fit_v6_stack_ridge(O_sub, meta_sub, y_sub, alpha=5.0):
    features = np.column_stack([O_sub, meta_sub])
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    model = Ridge(alpha=alpha)
    model.fit(features_scaled, y_sub)
    return model, scaler


def predict_v6_stack_ridge(model, scaler, O_sub, meta_sub):
    features = np.column_stack([O_sub, meta_sub])
    features_scaled = scaler.transform(features)
    return np.clip(model.predict(features_scaled), 0, None)


group_cv = GroupKFold(n_splits=5)
raw_group_oof = np.zeros(len(train))
final_group_oof = np.zeros(len(train))
fold_metrics_rows = []

for fold, (train_idx, valid_idx) in enumerate(
    group_cv.split(O_base, y_np, groups=train["source_id"]), start=1
):
    model, scaler = fit_v6_stack_ridge(
        O_base[train_idx], meta_extra[train_idx], y_np[train_idx]
    )
    train_raw = predict_v6_stack_ridge(model, scaler, O_base[train_idx], meta_extra[train_idx])
    valid_raw = predict_v6_stack_ridge(model, scaler, O_base[valid_idx], meta_extra[valid_idx])
    train_final = apply_tail_shrink(train_raw)
    valid_final = apply_tail_shrink(valid_raw)

    raw_group_oof[valid_idx] = valid_raw
    final_group_oof[valid_idx] = valid_final
    fold_metrics_rows.append({
        "fold": fold,
        "n_valid": len(valid_idx),
        "n_source_valid": train.iloc[valid_idx]["source_id"].nunique(),
        "train_RMSE": root_mean_squared_error(y_np[train_idx], train_final),
        "validation_RMSE": root_mean_squared_error(y_np[valid_idx], valid_final),
        "raw_validation_RMSE": root_mean_squared_error(y_np[valid_idx], valid_raw),
        "validation_MAE": mean_absolute_error(y_np[valid_idx], valid_final),
        "validation_R2": r2_score(y_np[valid_idx], valid_final),
    })

fold_metrics = pd.DataFrame(fold_metrics_rows).set_index("fold")
group_metrics = fold_metrics[[
    "n_valid", "n_source_valid", "raw_validation_RMSE",
    "validation_RMSE", "validation_MAE", "validation_R2",
]].copy()

oof_pred_raw = raw_group_oof
oof_pred = final_group_oof
raw_mean_group_rmse = float(fold_metrics["raw_validation_RMSE"].mean())
final_mean_group_rmse = float(fold_metrics["validation_RMSE"].mean())

raw_oof_rowwise_metrics = pd.Series(regression_metrics(y, oof_pred_raw), name=f"{RAW_ENSEMBLE_NAME} rowwise")
oof_rowwise_metrics = pd.Series(regression_metrics(y, oof_pred), name=f"{FINAL_MODEL_NAME} rowwise")

raw_oof_metrics = raw_oof_rowwise_metrics.copy()
raw_oof_metrics.name = RAW_ENSEMBLE_NAME
raw_oof_metrics["RMSE"] = raw_mean_group_rmse

oof_metrics = oof_rowwise_metrics.copy()
oof_metrics.name = FINAL_MODEL_NAME
oof_metrics["RMSE"] = final_mean_group_rmse
group_overall = oof_metrics.copy()

tail_oof_comparison = pd.DataFrame([
    {
        "model": RAW_ENSEMBLE_NAME,
        "mean_groupfold_RMSE": raw_mean_group_rmse,
        "rowwise_RMSE": raw_oof_rowwise_metrics["RMSE"],
        **{k: v for k, v in raw_oof_rowwise_metrics.to_dict().items() if k != "RMSE"},
    },
    {
        "model": FINAL_MODEL_NAME,
        "mean_groupfold_RMSE": final_mean_group_rmse,
        "rowwise_RMSE": oof_rowwise_metrics["RMSE"],
        **{k: v for k, v in oof_rowwise_metrics.to_dict().items() if k != "RMSE"},
    },
]).set_index("model")

base_rows = []
for name, pred in base_oof.items():
    base_rows.append({
        "model": PRETTY_NAME.get(name, name),
        "validation_type": "stratified OOF",
        **regression_metrics(y, pred),
    })
base_rows.extend([
    {"model": RAW_ENSEMBLE_NAME, "validation_type": "source_id GroupKFold", **raw_oof_metrics.to_dict()},
    {"model": FINAL_MODEL_NAME, "validation_type": "source_id GroupKFold + final correction", **oof_metrics.to_dict()},
])
model_comparison = pd.DataFrame(base_rows).set_index("model").sort_values("RMSE")

chosen_rows = v6_summary[v6_summary["name"].str.contains("CHOSEN", na=False)]
best_strategy = (
    chosen_rows["name"].iloc[-1].replace("ensemble_v6_CHOSEN_", "")
    if len(chosen_rows) else "stack_ridge"
)
best_strategy_rmse = float(chosen_rows["rmse"].iloc[-1]) if len(chosen_rows) else np.nan

print(f"Loaded base models: {', '.join(base_model_names)}")
print(f"Selected ensemble strategy: {best_strategy}")
print(f"Best competition score used as external reference: {PUBLIC_BEST_SCORE:.5f}")


In [ ]:
print("Model summary:")
display(v6_summary)

print("Grouped validation metrics for the selected stacked model:")
display(group_metrics)

print("Before vs after high-tail correction:")
display(tail_oof_comparison)

print("Base and final model comparison:")
display(model_comparison)


In [ ]:
residuals = y - oof_pred
band_b_available = train[band_b_cols].notna().any(axis=1)
band_b_metrics = pd.DataFrame([
    {
        "kelompok": "Band B tersedia",
        "n": band_b_available.sum(),
        **regression_metrics(y[band_b_available], oof_pred[band_b_available]),
    },
    {
        "kelompok": "Band B tidak tersedia",
        "n": (~band_b_available).sum(),
        **regression_metrics(y[~band_b_available], oof_pred[~band_b_available]),
    },
]).set_index("kelompok")
display(band_b_metrics)

_, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(fold_metrics.index, fold_metrics["train_RMSE"], marker="o", label="Train", color=BLUE)
axes[0].plot(fold_metrics.index, fold_metrics["validation_RMSE"], marker="o", label="Validation", color=RED)
axes[0].set_title("Stacked model RMSE per source_id fold")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("RMSE")
axes[0].legend()

axes[1].scatter(y, oof_pred, alpha=0.22, s=14, color=GREEN)
lims = [min(y.min(), oof_pred.min()), max(y.max(), oof_pred.max())]
axes[1].plot(lims, lims, "--", color=RED)
axes[1].set_title("OOF aktual vs prediksi model final")
axes[1].set_xlabel("Aktual")
axes[1].set_ylabel("Prediksi")

axes[2].scatter(oof_pred, residuals, alpha=0.22, s=14, color=LIGHT_GREEN)
axes[2].axhline(0, ls="--", color=RED)
axes[2].set_title("Residual vs prediksi model final")
axes[2].set_xlabel("Prediksi")
axes[2].set_ylabel("Aktual - prediksi")
plt.tight_layout()
plt.show()


In [ ]:
mean_train_rmse = fold_metrics["train_RMSE"].mean()
mean_valid_rmse = fold_metrics["validation_RMSE"].mean()
generalization_gap = mean_valid_rmse - mean_train_rmse
gap_pct = generalization_gap / mean_valid_rmse * 100

base_only = model_comparison[model_comparison["validation_type"].eq("stratified OOF")]
best_base_name = base_only.index[0]
best_base_rmse = base_only.iloc[0]["RMSE"]

if generalization_gap > 4:
    diagnosis = "overfit cukup jelas pada meta-model, tetapi validasi berbasis sumber tetap menjadi acuan utama"
elif generalization_gap > 1:
    diagnosis = "overfit ringan hingga moderat"
else:
    diagnosis = "tidak menunjukkan overfit kuat pada validasi berbasis sumber"

display(Markdown(f'''
## Jawaban Soal 2

Model final menunjukkan **{diagnosis}**, bukan underfit.

- Mean train RMSE: **{mean_train_rmse:.3f}**
- Mean validation RMSE berbasis `source_id`: **{mean_valid_rmse:.3f}**
- Generalization gap: **{generalization_gap:.3f}** atau **{gap_pct:.1f}%**
- RMSE sebelum koreksi ekor tinggi: **{raw_oof_metrics["RMSE"]:.3f}**
- RMSE setelah koreksi ekor tinggi: **{oof_metrics["RMSE"]:.3f}**
- Base model tunggal terbaik: **{best_base_name}** dengan RMSE **{best_base_rmse:.3f}**

Performa jauh lebih baik dari prediksi konstan dan R2 OOF yang substantif menunjukkan model tidak
underfit. Risiko utama adalah **domain shift antar `source_id`**, bukan kurangnya kapasitas model.
Karena itu, validasi berbasis sumber lebih dipercaya daripada random CV untuk menilai kemampuan generalisasi
ke sumber atau laboratorium baru.

**Mitigasi yang digunakan:** target encoding dibuat fold-safe, `source_id` disusutkan ke prior regional,
model dibuat beragam (CatBoost, LightGBM, XGBoost, Ridge, dan HGB), regularisasi diterapkan pada model
boosting, dan prediksi tinggi dikoreksi secara konservatif.

**Keterbatasan:** validasi berbasis `source_id` masih proxy. Untuk penggunaan operasional, model tetap perlu
diuji pada wilayah dan periode sampling yang benar-benar baru.
'''))


# Soal 3 — Distribusi geografis dan ekosistem dominan

> **Apakah ada pola distribusi kandungan organik tanah yang berbeda secara signifikan antar wilayah
> geografis? Berdasarkan pola tersebut, kira-kira kondisi ekosistem seperti apa yang paling dominan
> dalam dataset ini? Jelaskan reasoning Anda berdasarkan informasi ekologis dan tutupan lahan yang
> terlihat!**

## Pendekatan dan asumsi

Target sangat right-skewed, sehingga perbedaan zona macro diuji dengan Kruskal-Wallis. Epsilon-squared
digunakan sebagai ukuran efek. Jika uji global signifikan, pasangan zona diuji dengan Mann-Whitney U
dan p-value dikoreksi menggunakan metode Holm.

In [ ]:
macro_summary = (
    train.groupby("geo_zone_macro")[TARGET]
    .agg(count="size", mean="mean", median="median", std="std")
    .sort_values("mean", ascending=False)
)
macro_groups = [
    group[TARGET].values for _, group in train.groupby("geo_zone_macro")
]
kw = stats.kruskal(*macro_groups)
k_macro = train["geo_zone_macro"].nunique()
epsilon_squared = (kw.statistic - k_macro + 1) / (len(train) - k_macro)

posthoc_rows = []
for zone_a, zone_b in combinations(sorted(train["geo_zone_macro"].unique()), 2):
    a = train.loc[train["geo_zone_macro"] == zone_a, TARGET]
    b = train.loc[train["geo_zone_macro"] == zone_b, TARGET]
    result = stats.mannwhitneyu(a, b, alternative="two-sided")
    rank_biserial = 2 * result.statistic / (len(a) * len(b)) - 1
    posthoc_rows.append({
        "zona_A": zone_a,
        "zona_B": zone_b,
        "U": result.statistic,
        "p_raw": result.pvalue,
        "rank_biserial": rank_biserial,
    })
posthoc = pd.DataFrame(posthoc_rows)
posthoc["p_holm"] = holm_adjust(posthoc["p_raw"])
posthoc["signifikan_5pct"] = posthoc["p_holm"] < 0.05

display(macro_summary)
print(
    f"Kruskal-Wallis H={kw.statistic:.3f}, p={kw.pvalue:.3e}, "
    f"epsilon-squared={epsilon_squared:.3f}"
)
display(posthoc.sort_values("p_holm"))

In [ ]:
biome_land_mean = train.pivot_table(
    index="biome", columns="land_cover_type", values=TARGET, aggfunc="mean"
)
biome_land_count = train.pivot_table(
    index="biome", columns="land_cover_type", values=TARGET, aggfunc="size",
    fill_value=0
)

_, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.boxplot(
    data=train, x="geo_zone_macro", y=TARGET,
    order=macro_summary.index, showfliers=False,
    color=LIGHT_GREEN, ax=axes[0]
)
axes[0].set_title("Distribusi target per zona macro")
axes[0].set_xlabel("Zona macro")
axes[0].set_ylabel("Kandungan organik")

sns.heatmap(
    biome_land_count, cmap="YlGn", annot=True, fmt=".0f",
    linewidths=0.3, ax=axes[1], cbar_kws={"label": "Jumlah sampel"}
)
axes[1].set_title("Komposisi bioma × tutupan lahan")
axes[1].set_xlabel("Tutupan lahan")
axes[1].set_ylabel("Bioma")
axes[1].tick_params(axis="y", rotation=0)
axes[1].tick_params(axis="x", rotation=45)
plt.setp(
    axes[1].get_xticklabels(),
    ha="right",
    rotation_mode="anchor",
)
plt.tight_layout()
plt.show()

landcover_summary = (
    train.groupby("land_cover_type")[TARGET]
    .agg(count="size", mean="mean", median="median")
    .sort_values("count", ascending=False)
)
display(landcover_summary)

In [ ]:
dominant_biome = train["biome"].value_counts().idxmax()
dominant_cover = train["land_cover_type"].value_counts().idxmax()
highest_macro = macro_summary.index[0]
lowest_macro = macro_summary.index[-1]

display(Markdown(f'''
## Jawaban Soal 3

**Ada perbedaan geografis yang signifikan dan bermakna secara praktis.** Uji Kruskal-Wallis menghasilkan
H = **{kw.statistic:.2f}**, p-value **{kw.pvalue:.2e}**, dan epsilon-squared **{epsilon_squared:.3f}**.
Ukuran efek ini termasuk kecil hingga moderat: perbedaan nyata, tetapi sebagian besar variasi tetap
berada di dalam zona. Zona **{highest_macro}**
memiliki mean tertinggi (**{macro_summary.loc[highest_macro, "mean"]:.2f}**), sedangkan
**{lowest_macro}** terendah (**{macro_summary.loc[lowest_macro, "mean"]:.2f}**).

Post-hoc Holm menunjukkan pasangan mana yang tetap berbeda setelah mengendalikan multiple testing.
Effect size `rank_biserial` memberi konteks karena p-value dapat sangat kecil hanya akibat jumlah sampel
besar.

Berdasarkan **jumlah sampel**, ekosistem dominan adalah bioma **{dominant_biome}** dan tutupan lahan
**{dominant_cover}**. Kelompok besar lain adalah Cerrado/Savannah. Dengan demikian, dataset didominasi
mosaik hutan Atlantik musiman dan savana tropis Brasil, bukan satu tipe ekosistem homogen. Dominan secara
jumlah tidak sama dengan mean target tertinggi: tutupan dengan sampel sedikit dapat memiliki mean tinggi,
sehingga ukuran sampel harus selalu diperiksa.

Secara ekologis, variasi bahan organik dapat dipengaruhi masukan serasah, produktivitas, kelembapan,
musim kering, kebakaran, tekstur tanah, dan penggunaan lahan. Akan tetapi, dataset ini tidak mengukur
seluruh mekanisme tersebut sehingga interpretasinya tetap berupa hipotesis yang konsisten dengan pola.

Referensi konteks: [IBGE — Brazilian biomes](https://www.ibge.gov.br/en/geosciences/environmental-information/vegetation/18391-biomes.html);
[MapBiomas Brasil](https://brasil.mapbiomas.org/en/);
[IPCC Climate Change and Land](https://www.ipcc.ch/srccl/).

**Keterbatasan:** zona adalah kategori, bukan jarak spasial kontinu; komposisi `source_id` juga berbeda
antarwilayah sehingga sebagian perbedaan dapat merefleksikan protokol sampling atau laboratorium.
'''))

# Soal 4 — Korelasi antar tingkat wilayah dan rasio rata-rata

> **Apakah ada korelasi kandungan organik tanah antar tingkat wilayah geografis? Berapa rasio
> perbandingan rata-rata kandungan organik antar wilayah tersebut, dan bagaimana ini memengaruhi
> pendekatan pemodelan Anda?**

## Pendekatan dan asumsi

Untuk tiap level wilayah dihitung mean, median, ukuran sampel, dan bootstrap 95% CI. Rasio utama hanya
menggunakan kelompok dengan minimal 20 sampel agar tidak didominasi kelompok sangat kecil.

In [ ]:
geo_levels = ["geo_zone_macro", "geo_zone_meso", "geo_zone_micro"]
geo_detail_tables = {}
geo_summary_rows = []
geo_mean_encoded = pd.DataFrame(index=train.index)

for level_index, level in enumerate(geo_levels):
    rows = []
    for group_name, group in train.groupby(level):
        ci_low, ci_high = bootstrap_mean_ci(
            group[TARGET], n_boot=400,
            seed=RANDOM_STATE + level_index + len(group)
        )
        rows.append({
            "group": group_name,
            "count": len(group),
            "mean": group[TARGET].mean(),
            "median": group[TARGET].median(),
            "ci_low": ci_low,
            "ci_high": ci_high,
        })
    detail = pd.DataFrame(rows).sort_values("mean")
    stable = detail.query("count >= 20")
    geo_detail_tables[level] = detail
    geo_summary_rows.append({
        "level": level,
        "n_group": len(detail),
        "n_group_stable": len(stable),
        "min_group": stable.iloc[0]["group"],
        "min_mean": stable.iloc[0]["mean"],
        "max_group": stable.iloc[-1]["group"],
        "max_mean": stable.iloc[-1]["mean"],
        "max_min_ratio": stable.iloc[-1]["mean"] / stable.iloc[0]["mean"],
        "eta_squared": eta_squared(train, level),
    })
    group_means = train.groupby(level)[TARGET].mean()
    geo_mean_encoded[level] = train[level].map(group_means)

geo_summary = pd.DataFrame(geo_summary_rows).set_index("level")
geo_corr = geo_mean_encoded.corr(method="spearman")
display(geo_summary)
display(geo_corr)
display(geo_detail_tables["geo_zone_macro"])

In [ ]:
_, axes = plt.subplots(1, 2, figsize=(14, 5))
geo_summary["max_min_ratio"].plot(kind="bar", color=GREEN, ax=axes[0])
axes[0].set_title("Rasio mean maksimum/minimum (n ≥ 20)")
axes[0].set_ylabel("Rasio")
axes[0].tick_params(axis="x", rotation=20)

sns.heatmap(
    geo_corr, annot=True, fmt=".2f", cmap="YlGn", vmin=0, vmax=1,
    ax=axes[1]
)
axes[1].set_title("Korelasi Spearman mean kelompok yang dipetakan ke sampel")
axes[1].tick_params(axis="y", rotation=0)
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(f'''
## Jawaban Soal 4

Ada hubungan hierarkis positif, tetapi tidak sempurna. Korelasi Spearman mean kelompok yang dipetakan
kembali ke sampel adalah:

- macro–meso: **{geo_corr.loc["geo_zone_macro", "geo_zone_meso"]:.2f}**
- meso–micro: **{geo_corr.loc["geo_zone_meso", "geo_zone_micro"]:.2f}**
- macro–micro: **{geo_corr.loc["geo_zone_macro", "geo_zone_micro"]:.2f}**

Untuk kelompok dengan minimal 20 sampel, rasio mean maksimum/minimum adalah
**{geo_summary.loc["geo_zone_macro", "max_min_ratio"]:.2f}x** pada macro,
**{geo_summary.loc["geo_zone_meso", "max_min_ratio"]:.2f}x** pada meso, dan
**{geo_summary.loc["geo_zone_micro", "max_min_ratio"]:.2f}x** pada micro.
Rasio yang makin besar pada resolusi lokal menunjukkan heterogenitas lokal yang tidak tertangkap oleh
zona macro saja.

**Dampak pada modeling:** semua level dipertahankan dan digabungkan dalam fitur hierarki. Model nonlinear
dapat mempelajari interaksi antartingkat, tetapi evaluasi deployment perlu spatial/group validation.
Bootstrap CI dan `count` mencegah mean kelompok kecil dianggap sama kuatnya dengan kelompok besar.

**Keterbatasan:** korelasi ini adalah korelasi mean kategori yang dipetakan ke baris, bukan Moran's I atau
semivariogram. Koordinat banyak yang hilang, sehingga autokorelasi spasial kontinu tidak dapat diuji
secara lengkap.
'''))

# Soal 5a — Kondisi acidity dan kapasitas tukar kation

> **Berapa rata-rata kandungan organik tanah dari tiap ekosistem ketika tingkat keasaman tanah berada
> di bawah persentil 25 dan kapasitas tukar kation berada di bawah rata-rata? Apa insight yang bisa
> didapat dari hal ini?**

## Pendekatan dan asumsi

P25 acidity dan mean KTK dihitung dari nilai nonmissing. Kelompok ekosistem menggunakan kolom `biome`.
Selain mean, ditampilkan median, standard deviation, dan bootstrap 95% CI.

In [ ]:
acidity_p25 = train["property_acidity_index"].quantile(0.25)
cec_mean = train["cation_exchange_capacity"].mean()
subset_5a = train[
    (train["property_acidity_index"] < acidity_p25)
    & (train["cation_exchange_capacity"] < cec_mean)
].copy()

q5a_rows = []
for index, (biome, group) in enumerate(subset_5a.groupby("biome")):
    ci_low, ci_high = bootstrap_mean_ci(
        group[TARGET], n_boot=1000, seed=RANDOM_STATE + index
    )
    q5a_rows.append({
        "biome": biome,
        "count": len(group),
        "mean": group[TARGET].mean(),
        "median": group[TARGET].median(),
        "std": group[TARGET].std(),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "cukup_stabil_n_ge_20": len(group) >= 20,
    })
q5a = pd.DataFrame(q5a_rows).sort_values("mean", ascending=False)

print(f"P25 acidity index = {acidity_p25:.4f}")
print(f"Mean cation exchange capacity = {cec_mean:.4f}")
print(f"Jumlah sampel hasil filter = {len(subset_5a)}")
display(q5a)

stable_q5a = q5a.query("count >= 20").sort_values("mean")
plt.figure(figsize=(9, 4.8))
xerr = np.vstack([
    stable_q5a["mean"] - stable_q5a["ci_low"],
    stable_q5a["ci_high"] - stable_q5a["mean"],
])
plt.barh(stable_q5a["biome"], stable_q5a["mean"], color=GREEN, xerr=xerr)
plt.title("Mean dan bootstrap 95% CI (bioma dengan n ≥ 20)")
plt.xlabel("Kandungan organik rata-rata")
plt.tight_layout()
plt.show()

In [ ]:
stable_text = "; ".join(
    f"{row.biome}: mean {row.mean:.2f} (n={int(row.count)})"
    for row in stable_q5a.itertuples()
)
display(Markdown(f'''
## Jawaban Soal 5a

Ambang filter adalah acidity index < **{acidity_p25:.4f}** dan KTK <
**{cec_mean:.4f}**. Total sampel yang memenuhi kedua syarat adalah **{len(subset_5a)}**.

Untuk kelompok dengan ukuran memadai: **{stable_text}**. Bioma dengan n sangat kecil tetap ditampilkan
pada tabel, tetapi tidak dijadikan dasar generalisasi karena confidence interval atau mean-nya tidak
stabil.

**Insight:** acidity rendah dan KTK di bawah rata-rata tidak menghasilkan satu level kandungan organik
yang seragam. Perbedaan antarekosistem tetap terlihat, sehingga kondisi kimia ini perlu diinterpretasikan
bersama bioma, tekstur, tutupan lahan, dan sumber data. Mean juga perlu didampingi median dan CI karena
target right-skewed.

**Keterbatasan:** filter complete-case membuat hasil hanya mewakili sampel yang memiliki kedua pengukuran.
Dengan missingness acidity yang tinggi, subset ini dapat memiliki selection bias.
'''))

# Soal 5b — Outlier kombinasi tutupan lahan dan wilayah

> **Apakah ada kombinasi jenis tutupan lahan dan wilayah geografis tertentu yang memiliki nilai
> kandungan organik yang dapat dianggap sebagai outlier? Jelaskan justifikasi Anda!**

## Pendekatan dan asumsi

Outlier dihitung **di dalam setiap kombinasi** `land_cover_type × geo_zone_macro` dengan minimal 20
sampel. Definisi utama menggunakan IQR; sensitivity check menggunakan robust z-score berbasis MAD.

In [ ]:
combo = train.groupby(["land_cover_type", "geo_zone_macro"])[TARGET].agg(
    n="size",
    mean="mean",
    median="median",
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
    mad=lambda s: stats.median_abs_deviation(s, scale=1, nan_policy="omit"),
).reset_index()
combo["iqr"] = combo["q3"] - combo["q1"]
combo["lower_iqr"] = combo["q1"] - 1.5 * combo["iqr"]
combo["upper_iqr"] = combo["q3"] + 1.5 * combo["iqr"]

outlier_data = train.merge(
    combo, on=["land_cover_type", "geo_zone_macro"], how="left"
)
outlier_data["outlier_iqr"] = (
    (outlier_data[TARGET] < outlier_data["lower_iqr"])
    | (outlier_data[TARGET] > outlier_data["upper_iqr"])
)
outlier_data["robust_z"] = np.where(
    outlier_data["mad"] > 0,
    0.6745 * (outlier_data[TARGET] - outlier_data["median"])
    / outlier_data["mad"],
    0,
)
outlier_data["outlier_mad"] = outlier_data["robust_z"].abs() > 3.5

outlier_summary = (
    outlier_data.groupby(["land_cover_type", "geo_zone_macro"])
    .agg(
        n=(TARGET, "size"),
        mean_target=(TARGET, "mean"),
        outlier_iqr=("outlier_iqr", "sum"),
        outlier_mad=("outlier_mad", "sum"),
        max_target=(TARGET, "max"),
    )
    .reset_index()
)
outlier_summary["iqr_rate"] = outlier_summary["outlier_iqr"] / outlier_summary["n"]
outlier_summary["mad_rate"] = outlier_summary["outlier_mad"] / outlier_summary["n"]
outlier_summary["kombinasi"] = (
    outlier_summary["land_cover_type"] + " | "
    + outlier_summary["geo_zone_macro"]
)
reliable_outliers = (
    outlier_summary.query("n >= 20")
    .sort_values(["iqr_rate", "outlier_iqr"], ascending=False)
)
display(reliable_outliers.head(15))

In [ ]:
top_outliers = reliable_outliers.head(10).sort_values("iqr_rate")
plt.figure(figsize=(11, 5.8))
plt.barh(
    top_outliers["kombinasi"], top_outliers["iqr_rate"] * 100,
    color=GREEN, label="IQR"
)
plt.scatter(
    top_outliers["mad_rate"] * 100,
    top_outliers["kombinasi"],
    color=RED, label="MAD robust z-score"
)
plt.title("Outlier rate per kombinasi (n ≥ 20)")
plt.xlabel("Outlier rate (%)")
plt.legend()
plt.tight_layout()
plt.show()

individual_outliers = (
    outlier_data.loc[
        outlier_data["outlier_iqr"] & (outlier_data["n"] >= 20),
        ["sample_id", "land_cover_type", "geo_zone_macro", TARGET,
         "lower_iqr", "upper_iqr", "robust_z"]
    ]
    .sort_values(TARGET, ascending=False)
    .head(15)
)
display(individual_outliers)

In [ ]:
top_combo = reliable_outliers.iloc[0]
display(Markdown(f'''
## Jawaban Soal 5b

Ya. Dengan aturan IQR di dalam kelompok dan minimum 20 sampel, kombinasi dengan proporsi outlier
tertinggi adalah **{top_combo["kombinasi"]}**, yaitu
**{top_combo["iqr_rate"] * 100:.1f}%** ({int(top_combo["outlier_iqr"])} dari
{int(top_combo["n"])} sampel). Tabel dan grafik menunjukkan kombinasi lain serta perbandingan dengan
robust z-score berbasis MAD.

**Justifikasi:** batas dibuat per kombinasi agar kelompok yang secara alami memiliki mean atau sebaran
tinggi tidak ditandai hanya karena berbeda dari populasi global. Minimum 20 sampel mencegah kuartil
kelompok sangat kecil menjadi tidak stabil. Kesepakatan IQR dan MAD memperkuat indikasi, sedangkan
ketidaksepakatan menunjukkan hasil sensitif terhadap definisi outlier.

Outlier tidak otomatis dihapus. Nilai ekstrem dapat merepresentasikan kondisi mikro-lokasi, tanah
organik, drainase, pengelolaan, atau perbedaan laboratorium. Langkah yang tepat adalah pemeriksaan
metadata, analisis residual, dan robust validation.

**Keterbatasan:** IQR/MAD bersifat univariat dan tidak menilai apakah kombinasi fitur lain membuat nilai
tersebut sebenarnya masuk akal.
'''))

# Soal 6 — Korelasi tinggi dan multicollinearity

> **Apakah ada pasangan variabel yang berkorelasi tinggi? Apakah ada efek multicollinearity yang perlu
> diatasi dalam modeling?**

## Pendekatan dan asumsi

Spearman digunakan karena distribusi tidak normal dan hubungan dapat monotonic nonlinear. VIF dihitung
pada fitur tanah utama setelah median imputation dan standardization.

In [ ]:
numeric = train.select_dtypes(include=np.number)
spearman = numeric.corr(method="spearman")
upper = np.triu(np.ones(spearman.shape), k=1).astype(bool)
corr_pairs = spearman.where(upper).stack().rename("spearman_r").to_frame()
corr_pairs["abs_r"] = corr_pairs["spearman_r"].abs()
high_corr_pairs = corr_pairs.query("abs_r >= 0.70").sort_values(
    "abs_r", ascending=False
)
display(high_corr_pairs)

selected_corr_features = [
    "property_particle_coarse", "property_particle_fine",
    "property_acidity_index", "cation_Ca", "cation_Mg",
    "cation_exchange_capacity", "spectral_band_A_PC_1",
    "spectral_band_A_PC_2", "spectral_band_A_PC_3",
    "spectral_band_A_PC_5", "spectral_band_A_PC_7",
    "spectral_band_A_PC_12", TARGET,
]
cluster = sns.clustermap(
    train[selected_corr_features].corr(method="spearman"),
    cmap="vlag", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", figsize=(11, 10)
)
cluster.fig.suptitle("Clustered Spearman correlation", y=1.02, fontweight="bold")
plt.show()

In [ ]:
vif_cols = [
    "property_particle_coarse", "property_particle_fine",
    "property_acidity_index", "cation_Ca", "cation_Mg",
    "cation_exchange_capacity",
]
vif_matrix = SimpleImputer(strategy="median").fit_transform(train[vif_cols])
vif_matrix = StandardScaler().fit_transform(vif_matrix)

vif_rows = []
for i, feature in enumerate(vif_cols):
    other = np.arange(len(vif_cols)) != i
    feature_r2 = LinearRegression().fit(
        vif_matrix[:, other], vif_matrix[:, i]
    ).score(vif_matrix[:, other], vif_matrix[:, i])
    vif_rows.append({
        "feature": feature,
        "VIF": 1 / (1 - feature_r2),
    })
vif_table = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
condition_number = np.linalg.cond(vif_matrix)
display(vif_table)
print(f"Condition number = {condition_number:.3f}")

In [ ]:
strongest_pair = high_corr_pairs.iloc[0]
display(Markdown(f'''
## Jawaban Soal 6

Ada pasangan dengan korelasi tinggi. Pasangan terkuat adalah
`{high_corr_pairs.index[0][0]}` dan `{high_corr_pairs.index[0][1]}` dengan Spearman
**{strongest_pair["spearman_r"]:.2f}**. Korelasi Ca–Mg juga tinggi. VIF tertinggi adalah
**{vif_table.iloc[0]["VIF"]:.2f}** pada `{vif_table.iloc[0]["feature"]}`.

Ini merupakan multikolinearitas moderat yang penting untuk regresi linear karena dapat membuat
koefisien tidak stabil dan sulit ditafsirkan. PLS menjadi pembanding relevan karena membentuk komponen
laten dari prediktor berkorelasi. CatBoost tidak membutuhkan inversi matriks dan lebih tahan terhadap
multikolinearitas, sehingga fitur tidak harus dibuang hanya berdasarkan VIF.

Dalam model utama, fraksi kasar/halus tetap dipertahankan dan ditambah fitur komposisional seperti
`fine_fraction`. Konsekuensinya, feature importance tidak boleh dibaca sebagai efek kausal individual
karena importance dapat terbagi di antara fitur yang saling berkorelasi.

Referensi PLS: Wold, Sjöström, & Eriksson (2001),
[PLS-regression: a basic tool of chemometrics](https://doi.org/10.1016/S0169-7439(01)00155-1).

**Keterbatasan:** VIF dihitung pada subset fitur tanah agar dapat ditafsirkan; VIF untuk seluruh dummy
kategori dan komponen engineered tidak akan memberi ringkasan yang stabil atau ringkas.
'''))

# Soal 7 - Feature engineering

> **Fitur baru apa saja yang Anda buat melalui proses feature engineering? Jelaskan bagaimana fitur
> tersebut meningkatkan pemahaman model terhadap pola kandungan organik tanah!**

## Pendekatan dan asumsi

Manfaat fitur diuji dengan ablation study bertahap pada split yang sama. Tujuannya bukan hanya membuat fitur sebanyak mungkin, tetapi melihat apakah kelompok fitur tertentu benar-benar membantu model membaca pola spektral, kimia tanah, missingness, dan konteks wilayah.


In [ ]:
split_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
idx_train, idx_valid = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=split_bins,
)

stage_labels = {
    "raw": "Raw",
    "missingness": "+ Missingness",
    "spectral": "+ Spectral summaries",
    "soil": "+ Soil and chemistry ratios",
    "geographic": "+ Coordinates and geo/ecology interactions",
    "full": "+ Frequency features (full)",
}
ablation_rows = []

for stage, label in stage_labels.items():
    X_stage, cat_stage = make_features(train, stage=stage)
    stage_model = CatBoostRegressor(
        iterations=400,
        depth=6,
        learning_rate=0.06,
        l2_leaf_reg=8,
        random_strength=1,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    stage_model.fit(
        X_stage.iloc[idx_train], y.iloc[idx_train],
        cat_features=cat_stage,
        eval_set=(X_stage.iloc[idx_valid], y.iloc[idx_valid]),
        early_stopping_rounds=70,
        verbose=False,
    )
    pred = stage_model.predict(X_stage.iloc[idx_valid])
    ablation_rows.append({
        "stage": stage,
        "label": label,
        "n_features": X_stage.shape[1],
        "best_iteration": stage_model.get_best_iteration(),
        "RMSE": root_mean_squared_error(y.iloc[idx_valid], pred),
        "MAE": mean_absolute_error(y.iloc[idx_valid], pred),
    })
    print(f"Ablation {stage} selesai.")

X_no_source, cat_no_source = make_features(
    train, stage="full", drop_source=True
)
no_source_model = CatBoostRegressor(
    iterations=400, depth=6, learning_rate=0.06, l2_leaf_reg=8,
    random_strength=1, loss_function="RMSE", random_seed=RANDOM_STATE,
    verbose=False, allow_writing_files=False, thread_count=-1,
)
no_source_model.fit(
    X_no_source.iloc[idx_train], y.iloc[idx_train],
    cat_features=cat_no_source,
    eval_set=(X_no_source.iloc[idx_valid], y.iloc[idx_valid]),
    early_stopping_rounds=70, verbose=False,
)
no_source_pred = no_source_model.predict(X_no_source.iloc[idx_valid])

ablation = pd.DataFrame(ablation_rows)
raw_rmse = ablation.loc[ablation["stage"] == "raw", "RMSE"].iloc[0]
ablation["RMSE_improvement_vs_raw"] = raw_rmse - ablation["RMSE"]
source_sensitivity = pd.Series({
    "full_RMSE": ablation.loc[ablation["stage"] == "full", "RMSE"].iloc[0],
    "without_source_RMSE": root_mean_squared_error(
        y.iloc[idx_valid], no_source_pred
    ),
})
source_sensitivity["delta_without_source"] = (
    source_sensitivity["without_source_RMSE"]
    - source_sensitivity["full_RMSE"]
)
display(ablation)
display(source_sensitivity.to_frame("nilai"))

In [ ]:
_, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(
    ablation["label"], ablation["RMSE"], marker="o", color=GREEN, lw=2
)
axes[0].set_title("Ablation study: validation RMSE")
axes[0].set_ylabel("RMSE (lebih rendah lebih baik)")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(
    ablation["label"], ablation["RMSE_improvement_vs_raw"],
    color=LIGHT_GREEN
)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("Perbaikan RMSE terhadap raw features")
axes[1].set_ylabel("Delta RMSE")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
best_stage = ablation.sort_values("RMSE").iloc[0]
display(Markdown(f'''
## Jawaban Soal 7

Fitur baru dibagi menjadi:

1. **Missingness:** jumlah nilai hilang, ketersediaan band B aktual, jumlah missing band B, dan
   ketersediaan koordinat.
2. **Ringkasan spektral:** mean, standard deviation, min, max, L2 norm, maximum absolute component,
   absolute-sum, jumlah komponen positif, serta transformasi absolute/squared untuk PC awal.
3. **Tekstur dan kimia:** total partikel, fine fraction, fine/coarse ratio, selisih fine-coarse,
   jumlah kation basa, rasio Ca/Mg dan Mg/Ca, proxy base saturation, rasio Ca/CEC, Mg/CEC,
   CEC per fine fraction, interaksi acidity x CEC, dan transformasi log1p untuk variabel kimia utama.
4. **Koordinat dan interaksi geografis/ekologis:** absolute latitude/longitude, kombinasi lat-lon,
   grid koordinat, hierarchy macro-meso-micro, biome-land cover, source-micro, source-landcover,
   meso-landcover, landcover-rock, source-rock, macro-biome, dan macro-landcover.
5. **Encoding kategori:** frequency encoding untuk kategori, lalu ditambah
   fold-safe target encoding dan anchored `source_id` target encoding menuju regional prior.

Stage terbaik pada holdout yang sama adalah **{best_stage["label"]}** dengan RMSE
**{best_stage["RMSE"]:.3f}**, dibanding raw RMSE **{raw_rmse:.3f}**. Ablation memperlihatkan apakah
penambahan kelompok fitur benar-benar meningkatkan prediksi, bukan hanya terdengar masuk akal.

Menghapus `source_id` mengubah RMSE sebesar **{source_sensitivity["delta_without_source"]:.3f}**.
Jika performa memburuk cukup besar, model menggunakan informasi domain/laboratorium; hal ini berguna
untuk prediksi, tetapi harus diaudit pada sumber baru. Karena itu, `source_id` tidak hanya dipakai mentah,
melainkan disusutkan ke prior regional agar source kecil tidak terlalu mendominasi.

**Bagaimana fitur membantu pemahaman model:** ringkasan spektral menangkap energi dan penyebaran sinyal;
rasio tanah merepresentasikan komposisi; interaksi kategori menangkap konteks lokal; indikator missingness
memberi informasi tentang proses pengukuran. Fitur geografis dan biome-land cover membantu model
membedakan pola SOC yang sama-sama ekologis tetapi berbeda konteks wilayah.

**Keterbatasan:** ablation satu holdout lebih cepat dan konsisten, tetapi memiliki uncertainty lebih besar
daripada nested cross-validation. Target encoding tidak dimasukkan ke ablation sederhana ini karena harus
dibangun fold-by-fold untuk menghindari leakage.
'''))


# Soal 8 - Model prediksi dan alasan pemilihan

> **Jelaskan model yang Anda gunakan dalam memprediksi kandungan organik tanah! Mengapa Anda memilih
> model tersebut dibanding alternatif lain, khususnya dalam konteks data yang memiliki representasi
> spektral?**

## Pendekatan dan asumsi

Model utama dibandingkan dengan baseline sederhana. Dummy median digunakan sebagai batas bawah, PLSRegression digunakan sebagai pembanding klasik untuk data spektral yang kolinear, lalu beberapa model nonlinear dibandingkan sebelum prediksinya digabungkan.

Alasan pemilihan model dilihat dari dua sisi: akurasi validasi dan kecocokan metode terhadap karakter data, terutama kombinasi fitur spektral, kimia tanah, kategori wilayah, missingness, dan potensi perbedaan antar sumber data.


In [ ]:
model_oof = {}
model_times = {}

model_oof["Dummy median"] = np.zeros(len(train))
model_oof["PLS numeric"] = np.zeros(len(train))
model_times["Dummy median"] = []
model_times["PLS numeric"] = []

numeric_full_cols = X_full.select_dtypes(include=np.number).columns.tolist()

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    dummy = DummyRegressor(strategy="median")
    started = time.perf_counter()
    dummy.fit(np.zeros((len(train_idx), 1)), y.iloc[train_idx])
    model_oof["Dummy median"][valid_idx] = dummy.predict(np.zeros((len(valid_idx), 1)))
    model_times["Dummy median"].append(time.perf_counter() - started)

    pls = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", PLSRegression(n_components=15, scale=False, max_iter=500)),
    ])
    started = time.perf_counter()
    pls.fit(X_full.iloc[train_idx][numeric_full_cols], y.iloc[train_idx])
    model_oof["PLS numeric"][valid_idx] = np.asarray(
        pls.predict(X_full.iloc[valid_idx][numeric_full_cols])
    ).ravel()
    model_times["PLS numeric"].append(time.perf_counter() - started)
    print(f"Benchmark baseline fold {fold} selesai.")

for name, pred in base_oof.items():
    label = PRETTY_NAME.get(name, name)
    model_oof[label] = pred.copy()
    model_times[label] = [np.nan] * len(cv_splits)

model_oof[RAW_ENSEMBLE_NAME] = oof_pred_raw.copy()
model_oof[FINAL_MODEL_NAME] = oof_pred.copy()
model_times[RAW_ENSEMBLE_NAME] = [np.nan] * len(cv_splits)
model_times[FINAL_MODEL_NAME] = [np.nan] * len(cv_splits)

benchmark_fold_rows = []
for fold, (_, valid_idx) in enumerate(cv_splits, start=1):
    for model_name, pred in model_oof.items():
        fold_result = regression_metrics(y.iloc[valid_idx], pred[valid_idx])
        benchmark_fold_rows.append({
            "fold": fold,
            "model": model_name,
            **fold_result,
            "train_seconds": model_times[model_name][fold - 1],
        })
benchmark_by_fold = pd.DataFrame(benchmark_fold_rows)
benchmark_cv_summary = benchmark_by_fold.groupby("model").agg(
    RMSE_mean=("RMSE", "mean"),
    RMSE_std=("RMSE", "std"),
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    R2_mean=("R2", "mean"),
    R2_std=("R2", "std"),
    mean_train_seconds=("train_seconds", "mean"),
).sort_values("RMSE_mean")

benchmark_rows = []
for model_name, pred in model_oof.items():
    times = np.asarray(model_times[model_name], dtype=float)
    mean_seconds = np.nan if np.isnan(times).all() else np.nanmean(times)
    benchmark_rows.append({
        "model": model_name,
        **regression_metrics(y, pred),
        "mean_train_seconds_per_fold": mean_seconds,
    })
benchmark = pd.DataFrame(benchmark_rows).set_index("model")
benchmark.loc[RAW_ENSEMBLE_NAME, "RMSE"] = raw_mean_group_rmse
benchmark.loc[FINAL_MODEL_NAME, "RMSE"] = final_mean_group_rmse
benchmark = benchmark.sort_values("RMSE")
print("Ringkasan metrik per fold atau subset fold research:")
display(benchmark_cv_summary)
print("Metrik gabungan seluruh prediksi OOF; RMSE model final memakai mean GroupKFold:")
display(benchmark)


In [ ]:
_, axes = plt.subplots(1, 2, figsize=(14, 5))
benchmark["RMSE"].sort_values().plot(kind="bar", color=GREEN, ax=axes[0])
axes[0].set_title("Perbandingan RMSE")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=35)

benchmark["MAE"].sort_values().plot(kind="bar", color=LIGHT_GREEN, ax=axes[1])
axes[1].set_title("Perbandingan MAE")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

print("Ringkasan model utama:")
display(v6_summary)


In [ ]:
best_model_name = benchmark.index[0]
final_rmse = benchmark.loc[FINAL_MODEL_NAME, "RMSE"]
raw_stack_rmse = benchmark.loc[RAW_ENSEMBLE_NAME, "RMSE"]
best_base_name = base_only.index[0]
best_base_rmse = base_only.iloc[0]["RMSE"]
final_delta_vs_best_base = final_rmse - best_base_rmse

display(Markdown(f'''
## Jawaban Soal 8

Model yang digunakan adalah **stacked ensemble**. Base modelnya terdiri dari CatBoost, LightGBM,
XGBoost, Ridge, dan HistGradientBoosting. Prediksi dari model-model tersebut digabung dengan stack Ridge
dan dua meta-fitur tambahan: frekuensi `source_id` serta total missing value. Strategi ensemble dipilih
dengan validasi berbasis `source_id`.

Perbandingan ringkas:

- Stacked model sebelum koreksi ekor tinggi: RMSE **{raw_stack_rmse:.3f}**
- Stacked model setelah koreksi ekor tinggi: RMSE **{final_rmse:.3f}**
- Base model tunggal terbaik: **{best_base_name}**, RMSE **{best_base_rmse:.3f}**
- PLS numeric: RMSE **{benchmark.loc["PLS numeric", "RMSE"]:.3f}**
- Dummy median: RMSE **{benchmark.loc["Dummy median", "RMSE"]:.3f}**

Model final dipilih bukan karena satu model tunggal selalu menang pada random OOF, tetapi karena ensemble
multi-family lebih tahan terhadap pola error yang berbeda dan validasi berbasis `source_id` lebih dekat
dengan skenario sumber data baru. PLS tetap relevan sebagai baseline chemometrics untuk prediktor spektral yang
kolinear, tetapi model final membutuhkan interaksi nonlinear antara spektrum, kimia tanah, missingness,
dan konteks geografis.

Selisih model final terhadap base OOF terbaik adalah **{final_delta_vs_best_base:.3f}**. Angka ini tidak
boleh dibaca sebagai perbandingan langsung murni karena base model dan stacked model dievaluasi dengan
skema validasi yang berbeda. Kesimpulan utamanya adalah ensemble memberi cara yang lebih stabil untuk
menggabungkan kekuatan beberapa keluarga model.

Referensi:
[Prokhorenkova et al. - CatBoost](https://arxiv.org/abs/1706.09516);
[Chen and Guestrin - XGBoost](https://doi.org/10.1145/2939672.2939785);
[Ke et al. - LightGBM](https://papers.nips.cc/paper_files/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html);
[Wold et al. - PLS regression](https://doi.org/10.1016/S0169-7439(01)00155-1);
[Viscarra Rossel et al. - diffuse reflectance soil spectroscopy](https://doi.org/10.1016/j.geoderma.2005.03.007).

**Keterbatasan:** benchmark ini bukan hyperparameter search menyeluruh. Hasilnya dipakai untuk menjelaskan
alasan pemilihan model, bukan untuk mengklaim bahwa semua alternatif sudah dieksplorasi habis.
'''))


# Soal 9 - Apakah RMSE tepat?

> **Menurut Anda, apakah metrik penilaian RMSE tepat untuk kasus prediksi kandungan organik tanah ini?
> Jika tidak, metrik apa yang lebih tepat? Elaborasikan jawaban Anda dengan mempertimbangkan
> karakteristik distribusi target!**

## Pendekatan dan asumsi

RMSE dievaluasi bersama MAE, MedianAE, dan RMSLE. Karena target memiliki ekor kanan yang panjang, analisis juga melihat kontribusi error per desil target untuk mengetahui apakah nilai organik tinggi terlalu mendominasi skor.


In [ ]:
metric_comparison = pd.Series(regression_metrics(y, oof_pred), name="nilai")
metric_comparison["Rowwise_RMSE"] = metric_comparison["RMSE"]
metric_comparison["RMSE"] = final_mean_group_rmse
error_analysis = pd.DataFrame({
    "actual": y,
    "prediction": oof_pred,
})
error_analysis["absolute_error"] = (
    error_analysis["actual"] - error_analysis["prediction"]
).abs()
error_analysis["squared_error"] = (
    error_analysis["actual"] - error_analysis["prediction"]
) ** 2
error_analysis["target_decile"] = pd.qcut(
    error_analysis["actual"], q=10, duplicates="drop"
)
decile_error = error_analysis.groupby(
    "target_decile", observed=True
).agg(
    n=("actual", "size"),
    target_mean=("actual", "mean"),
    MAE=("absolute_error", "mean"),
    MSE=("squared_error", "mean"),
    squared_error_sum=("squared_error", "sum"),
)
decile_error["share_total_squared_error"] = (
    decile_error["squared_error_sum"]
    / decile_error["squared_error_sum"].sum()
)

display(metric_comparison.to_frame())
display(decile_error)

_, axes = plt.subplots(1, 2, figsize=(15, 5))
decile_labels = [str(i + 1) for i in range(len(decile_error))]
axes[0].bar(decile_labels, decile_error["MAE"], color=LIGHT_GREEN)
axes[0].set_title("MAE per desil target")
axes[0].set_xlabel("Desil target (rendah → tinggi)")
axes[0].set_ylabel("MAE")

axes[1].bar(
    decile_labels, decile_error["share_total_squared_error"] * 100,
    color=GREEN
)
axes[1].set_title("Kontribusi terhadap total squared error")
axes[1].set_xlabel("Desil target (rendah → tinggi)")
axes[1].set_ylabel("Kontribusi (%)")
plt.tight_layout()
plt.show()

In [ ]:
top_decile_share = decile_error.iloc[-1]["share_total_squared_error"] * 100
display(Markdown(f'''
## Jawaban Soal 9

**RMSE tepat sebagai metrik utama kompetisi apabila kesalahan besar dianggap jauh lebih mahal**, tetapi
tidak cukup sebagai satu-satunya metrik. Target memiliki skewness **{y.skew():.2f}** dengan ekor kanan
panjang. RMSE validasi berbasis sumber adalah **{metric_comparison["RMSE"]:.3f}**, sedangkan MAE
**{metric_comparison["MAE"]:.3f}**, MedianAE **{metric_comparison["MedianAE"]:.3f}**, dan RMSLE
**{metric_comparison["RMSLE"]:.3f}**.

Desil target tertinggi menyumbang sekitar **{top_decile_share:.1f}%** dari total squared error. Ini
menunjukkan RMSE sangat sensitif terhadap sampel bernilai tinggi dan outlier. Sensitivitas tersebut dapat
diinginkan bila kegagalan pada tanah berkandungan sangat tinggi memiliki konsekuensi besar, tetapi dapat
membuat performa mayoritas sampel kurang terlihat.

**Rekomendasi:** pertahankan RMSE untuk konsistensi kompetisi, lalu dampingi dengan:

- MAE untuk error absolut rata-rata yang lebih robust;
- MedianAE untuk pengalaman sampel tipikal;
- RMSLE bila error relatif lebih penting;
- error per zona, bioma, ketersediaan band B, dan desil target untuk fairness/robustness.

Jadi, bukan mengganti RMSE sepenuhnya, melainkan menggunakan *metric portfolio* agar keputusan tidak
didominasi satu karakteristik distribusi.

**Keterbatasan:** pilihan metrik ideal seharusnya diturunkan dari biaya kesalahan operasional. Dataset
tidak menyediakan cost matrix atau ambang agronomis.
'''))

# Soal 10 — Data eksternal

> **Jika Anda boleh mengambil data eksternal, data tentang apa yang akan Anda ambil untuk meningkatkan
> akurasi prediksi kandungan organik tanah? Jelaskan alasan pemilihannya dan bagaimana data tersebut
> dapat diintegrasikan ke dalam pipeline pemodelan Anda!**

## Pendekatan dan asumsi

Prioritas dipilih berdasarkan proses pembentuk bahan organik: iklim, topografi, vegetasi/produktivitas,
penggunaan lahan, dan sifat tanah. Integrasi harus menggunakan lokasi dan informasi yang tersedia sebelum
tanggal sampling untuk mencegah leakage.

In [ ]:
external_data_plan = pd.DataFrame([
    {
        "sumber": "ERA5-Land",
        "variabel": "rainfall, temperature, soil moisture, evapotranspiration",
        "nilai_tambah": "menggambarkan produksi biomassa dan laju dekomposisi",
        "integrasi": "spatial join + agregasi 1/3/5 tahun sebelum sampling",
        "risiko": "resolusi lebih kasar dari titik tanah",
    },
    {
        "sumber": "Copernicus DEM / SRTM",
        "variabel": "elevation, slope, aspect, curvature, wetness index",
        "nilai_tambah": "menangkap drainase, erosi, dan akumulasi material",
        "integrasi": "extract raster dan terrain derivatives pada koordinat",
        "risiko": "koordinat hilang pada banyak sampel",
    },
    {
        "sumber": "Sentinel-2 / Landsat",
        "variabel": "NDVI/EVI, bare-soil index, moisture, burn history",
        "nilai_tambah": "proxy vegetasi, residu, gangguan, dan perubahan tutupan",
        "integrasi": "cloud masking + seasonal composites sebelum sampling",
        "risiko": "awan, tanggal sampling, dan mixed pixels",
    },
    {
        "sumber": "SoilGrids",
        "variabel": "clay, silt, sand, bulk density, SOC, pH",
        "nilai_tambah": "prior spasial sifat tanah global",
        "integrasi": "extract per depth yang sesuai 0–20 cm",
        "risiko": "target leakage bila memakai layer SOC kontemporer tanpa audit",
    },
    {
        "sumber": "MapBiomas Brasil",
        "variabel": "land-cover history, transition, fire, agriculture/pasture age",
        "nilai_tambah": "riwayat penggunaan lahan lebih informatif dari label snapshot",
        "integrasi": "annual history + transition counts sebelum sampling",
        "risiko": "ketidakselarasan tahun dan klasifikasi",
    },
])
display(external_data_plan)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")
sources = [
    (0.03, 0.78, "Iklim", "#A8DADC"),
    (0.03, 0.58, "Topografi", "#BDE0FE"),
    (0.03, 0.38, "Satelit", "#CDEAC0"),
    (0.03, 0.18, "Soil maps", "#E9C46A"),
]
for x, y_pos, label, color in sources:
    ax.text(x, y_pos, label, va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", fc=color, ec=DARK_GREEN))
    ax.annotate("", xy=(0.29, 0.5), xytext=(0.15, y_pos),
                arrowprops=dict(arrowstyle="->", color=DARK_GREEN))
pipeline_boxes = [
    (0.30, "Spatiotemporal\njoin"),
    (0.49, "Feature store +\nleakage checks"),
    (0.69, "Spatial/group\nvalidation"),
    (0.87, "Model +\nuncertainty"),
]
for x, label in pipeline_boxes:
    ax.text(x, 0.5, label, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.55", fc=LIGHT_GREEN, ec=DARK_GREEN))
for x1, x2 in [(0.37, 0.43), (0.57, 0.63), (0.77, 0.82)]:
    ax.annotate("", xy=(x2, 0.5), xytext=(x1, 0.5),
                arrowprops=dict(arrowstyle="->", lw=2, color=DARK_GREEN))
ax.set_title("Pipeline integrasi data eksternal tanpa temporal leakage", pad=15)
plt.show()

## Jawaban Soal 10

Prioritas pertama adalah **iklim historis, topografi, riwayat vegetasi/tutupan lahan, dan sifat tanah resolusi tinggi**. Variabel tersebut mengisi mekanisme yang belum sepenuhnya terlihat dari spektrum dan kategori wilayah.

Cara integrasinya:

1. validasi kualitas koordinat dan sistem proyeksi;
2. spatial join raster ke titik sampling;
3. agregasi temporal hanya dari periode sebelum sampling;
4. rekayasa fitur tren, variasi musiman, dan perubahan tutupan lahan;
5. pencatatan sumber, resolusi, dan tanggal data;
6. evaluasi menggunakan validasi berbasis wilayah atau sumber;
7. ablation untuk memastikan setiap sumber data benar-benar meningkatkan generalisasi.

SoilGrids SOC perlu diperlakukan hati-hati karena dapat menjadi proxy target dan menimbulkan leakage. Lebih aman memulai dari clay, bulk density, pH, topografi, iklim, dan riwayat tutupan lahan. Jika layer SOC digunakan, tahun, sumber observasi, dan overlap dengan data training harus diaudit.

Sumber resmi:

- [ERA5-Land](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview)
- [Copernicus DEM](https://dataspace.copernicus.eu/explore-data/data-collections/copernicus-contributing-missions/collections-description/COP-DEM)
- [Sentinel-2](https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html)
- [SoilGrids](https://isric.org/explore/soilgrids)
- [MapBiomas Brasil](https://brasil.mapbiomas.org/en/)

**Keterbatasan:** sebagian besar koordinat pada dataset hilang. Data raster baru bisa dimanfaatkan penuh jika koordinat lebih lengkap atau ada kunci lokasi administratif yang dapat dipetakan.


## Executive summary

In [ ]:
display(Markdown(f'''
1. Prediksi SOC penting sebagai alat screening untuk mengarahkan biaya sampling dan tindakan konservasi.
2. Model final memakai stacked ensemble dengan base CatBoost, LightGBM, XGBoost, Ridge, dan HGB;
   validasi utama dipisahkan berdasarkan `source_id` agar lebih realistis untuk sumber baru.
3. Model tidak underfit; risiko utamanya adalah domain shift antar sumber data.
4. Distribusi antar zona macro berbeda signifikan dengan epsilon-squared **{epsilon_squared:.3f}**.
5. Rasio mean antar kelompok meningkat pada resolusi geografis yang lebih lokal.
6. Analisis acidity/KTK harus mempertimbangkan sample size dan missingness; outlier tidak otomatis salah.
7. Multikolinearitas utama muncul pada tekstur kasar-halus dan Ca-Mg.
8. Ablation menunjukkan kontribusi empiris kelompok feature engineering; performa final tetap mengandalkan
   frequency encoding, fold-safe target encoding, dan anchored `source_id` target encoding.
9. RMSE relevan, tetapi perlu didampingi MAE, MedianAE, RMSLE, dan audit per kelompok/desil target.
10. Data iklim, topografi, satelit, SoilGrids, dan MapBiomas paling menjanjikan bila leakage dikendalikan.
'''))


## Daftar referensi

1. FAO Global Soil Partnership. **Soil Organic Carbon**.
   https://www.fao.org/global-soil-partnership/areas-of-work/soil-organic-carbon/en/
2. IPCC. **Special Report on Climate Change and Land**.
   https://www.ipcc.ch/srccl/
3. Prokhorenkova, L., et al. (2018). **CatBoost: unbiased boosting with categorical features**.
   https://arxiv.org/abs/1706.09516
4. Chen, T., & Guestrin, C. (2016). **XGBoost: A Scalable Tree Boosting System**.
   https://doi.org/10.1145/2939672.2939785
5. Ke, G., et al. (2017). **LightGBM: A Highly Efficient Gradient Boosting Decision Tree**.
   https://papers.nips.cc/paper_files/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html
6. Wold, S., Sjostrom, M., & Eriksson, L. (2001). **PLS-regression: a basic tool of chemometrics**.
   https://doi.org/10.1016/S0169-7439(01)00155-1
7. Viscarra Rossel, R. A., et al. (2006). **Visible, near infrared, mid infrared or combined diffuse
   reflectance spectroscopy for simultaneous assessment of various soil properties**.
   https://doi.org/10.1016/j.geoderma.2005.03.007
8. Copernicus Climate Data Store. **ERA5-Land**.
   https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview
9. Copernicus Data Space Ecosystem. **Copernicus DEM**.
   https://dataspace.copernicus.eu/explore-data/data-collections/copernicus-contributing-missions/collections-description/COP-DEM
10. Copernicus Data Space Ecosystem. **Sentinel-2 documentation**.
    https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html
11. ISRIC. **SoilGrids - global gridded soil information**.
    https://isric.org/explore/soilgrids
12. MapBiomas Brasil. **Land cover and land use mapping**.
    https://brasil.mapbiomas.org/en/
13. IBGE. **Biomes of Brazil**.
    https://www.ibge.gov.br/en/geosciences/environmental-information/vegetation/18391-biomes.html
